In [ ]:
import pymc as pm
import numpy as np
import arviz as az
from scipy.signal import lfilter

np.random.seed(42)
T = 100
hist_rain = np.random.uniform(0, 20, T)
hist_flow = np.zeros(T)
hist_flow[0] = 10

#  Loop kept for step-wise data generation clarity. # wil switch to vector eventualy

for t in range(1, T):
    hist_flow[t] = 10 + 0.8 * (hist_flow[t-1] - 10) + 0.5 * hist_rain[t] + np.random.normal(0, 1)

# Mean center it (as strictly required for AR)
flow_c = hist_flow - hist_flow.mean()
rain_c = hist_rain - hist_rain.mean()

with pm.Model() as ar_model:
    # MutableData registration for out-of-sample forecasting and modularity
    flow_data = pm.MutableData("flow_data", flow_c[:-1])
    rain_data = pm.MutableData("rain_data", rain_c[1:])
    obs_data = pm.MutableData("obs_data", flow_c[1:])

    # Priors
    rho = pm.Normal("rho", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # AR(1) with exogenous variable (Rain)
    mu = rho * flow_data + beta * rain_data

    # Likelihood
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=obs_data)

    # Inference
    trace = pm.sample(1000, tune=1000, return_inferencedata=True, random_seed=42)